# 🗒️ Entornos Python con uv

Guía de configuración de entornos de trabajo reproducibles.  
Cubre instalación, entornos virtuales, kernels de Jupyter y workflow profesional.

---

## Objetivo

Establecer un flujo de trabajo Python ordenado y reproducible desde cero, usando `uv` como gestor unificado de paquetes y entornos.

## Al terminar esta guía vas a poder:

1. Instalar Python y `uv` en Windows, Linux o macOS
2. Crear entornos virtuales aislados por proyecto
3. Registrar kernels de Jupyter y limpiarlos correctamente
4. Auditar el estado del entorno de desarrollo en cualquier momento

---

## Índice

1. [Entornos virtuales — por qué importan](#1)
2. [Instalar Python](#2)
3. [Instalar uv](#3)
4. [Configurar Python global y por proyecto](#4)
5. [Crear entornos virtuales con uv](#5)
6. [Registrar el entorno en VS Code](#6)
7. [Registrar el entorno como kernel de Jupyter](#7)
8. [Workflow completo: crear, usar y limpiar sin dejar huérfanos](#8)
9. [Verificación final del sistema](#9)

---

## ◈ Microglosario

| Término | Qué es en lenguaje llano |
|---|---|
| **Entorno virtual** | Directorio aislado con su propio Python y sus propios paquetes. Lo que instalás ahí no interfiere con otros proyectos ni con el sistema operativo. |
| **Kernel** | El proceso Python que ejecuta el código de las celdas de un notebook. Cada kernel apunta a un intérprete específico. |
| **Kernel huérfano** | Kernel registrado cuyo entorno virtual fue eliminado. Aparece en la lista pero falla al ejecutar — origen de errores confusos. |
| **PATH** | Lista de directorios donde el sistema operativo busca comandos. Si `python` no está en el PATH, la terminal no lo encuentra. |
| **Intérprete** | El ejecutable Python que procesa y corre el código. Su ruta (`.venv/bin/python`) determina qué paquetes están disponibles. |
| **Gestor de paquetes** | Herramienta para instalar, actualizar y eliminar librerías. `uv` reemplaza a `pip`, `venv` y `pip-tools` con una interfaz unificada. |

<a id='1'></a>
## 🧩 1. Entornos virtuales — por qué importan

### Analogía

Pensá en un entorno virtual como un departamento en un edificio. Cada inquilino tiene su propia llave, sus propios muebles y sus propias reglas — lo que pasa en el 3B no afecta al 5A. Sin entornos virtuales, todos los proyectos comparten el mismo Python del sistema: cualquier cambio de uno puede romper lo que funciona para los demás.

### Dónde vive esto en el mundo real

Cualquier equipo de desarrollo profesional trabaja con múltiples proyectos en paralelo, cada uno con versiones distintas de las mismas librerías. Sin aislamiento, actualizar una dependencia para el proyecto nuevo rompe el que ya está en producción. El `.venv/` en la raíz de cada repositorio es una práctica estándar de la industria — la vas a encontrar en cualquier empresa tech. Usarlos desde el principio garantiza que el flujo de trabajo sea exactamente el mismo que en un equipo real.

---

**El problema sin entornos virtuales:** si instalás `pandas 1.5` para un proyecto y luego `pandas 2.0` para otro, ambos comparten el mismo Python del sistema. El segundo proyecto rompe al primero. Esto se llama *dependency hell* y es la causa de la mayoría de los errores del tipo «funciona en mi máquina».

**La solución:** cada proyecto vive en su propio entorno. Las versiones de sus paquetes no interfieren entre sí ni con el sistema.

```
mi-laptop/
├── Python del sistema  ← no se toca nunca
├── proyecto-nlp/
│   └── .venv/          ← entorno con transformers 4.x, torch 2.x
└── proyecto-vision/
    └── .venv/          ← entorno con opencv 4.x, mediapipe 0.x
```

**¿Por qué uv y no pip/venv/conda?**

| Herramienta | Velocidad | Reproducibilidad | Multiplataforma |
|---|---|---|---|
| pip + venv | lenta | media | sí |
| conda | muy lenta | alta | sí |
| **uv** | **10-100× más rápida** | **alta** | **sí** |

`uv` es un gestor de paquetes y entornos escrito en Rust, desarrollado por Astral. Reemplaza a `pip`, `venv`, `pip-tools` y parcialmente a `pyenv` con una interfaz unificada y velocidad muy superior.

<a id='2'></a>
## 🌱 2. Instalar Python

> **Decisión importante:** `uv` puede instalar y gestionar versiones de Python por sí mismo. Si vas a usar uv (como se recomienda en esta guía), no es estrictamente necesario instalar Python de forma separada. Aun así, conviene tener al menos una versión base instalada en el sistema.

### Windows

1. Ir a [python.org/downloads](https://www.python.org/downloads/) y descargar el instalador de Python 3.11 o superior.
2. Al instalar, tildar la opción **"Add Python to PATH"** — si no lo hacés, Python no va a ser encontrable desde la terminal.
3. Verificar la instalación abriendo PowerShell o CMD:

```powershell
python --version
# Resultado esperado: Python 3.11.x
```

### Linux (Ubuntu/Debian)

```bash
sudo apt update
sudo apt install python3 python3-pip python3-venv -y
python3 --version
```

### macOS

La forma recomendada es Homebrew. Si no lo tenés instalado, la instalación es un solo comando (ver [brew.sh](https://brew.sh)).

```bash
brew install python@3.11
python3 --version
```

> **Nota sobre versiones:** Esta guía usa Python 3.11 como ejemplo por estabilidad y compatibilidad con la mayoría de las librerías de ciencia de datos y machine learning a la fecha (2025-2026). Podés reemplazar `3.11` por la versión que necesite tu proyecto.

<a id='3'></a>
## ⚡ 3. Instalar uv

`uv` tiene un instalador oficial que funciona en los tres sistemas operativos con un solo comando.

### Windows (PowerShell)

```powershell
powershell -ExecutionPolicy ByPass -c "irm https://astral.sh/uv/install.ps1 | iex"
```

Después de instalar, cerrar y reabrir PowerShell para que el PATH se actualice.

### Linux y macOS (terminal)

```bash
curl -LsSf https://astral.sh/uv/install.sh | sh
```

En macOS y Linux, el instalador modifica automáticamente el archivo de configuración del shell (`~/.bashrc`, `~/.zshrc`, etc.). Reiniciar la terminal o ejecutar:

```bash
source ~/.zshrc   # zsh (macOS por defecto)
# o
source ~/.bashrc  # bash (Linux)
```

### Verificar la instalación

En cualquier sistema:

```bash
uv --version
# Resultado esperado: uv 0.x.x
```

### Ver los intérpretes de Python que uv detecta

```bash
uv python list
```

Si uv no encontrás la versión que necesitás, uv puede descargarla e instalarla por su cuenta:

```bash
uv python install 3.11
```

<a id='4'></a>
## ⚙️ 4. Configurar Python global y por proyecto

Una práctica profesional fundamental es separar dos niveles de configuración:

- **Python global:** la versión que uv usa por defecto cuando no hay una instrucción específica. Es el «piso» del sistema.
- **Python por proyecto:** cada proyecto puede declarar la versión exacta que necesita, de forma independiente del global.

### Configurar el Python global

**Opción A — archivo de configuración de uv (recomendada)**

Crear o editar el archivo `~/.config/uv/uv.toml` (Linux/macOS) o `%APPDATA%\uv\uv.toml` (Windows):

```bash
# Linux / macOS
mkdir -p ~/.config/uv
echo 'python = "3.11"' >> ~/.config/uv/uv.toml
```

```powershell
# Windows PowerShell
New-Item -ItemType Directory -Force -Path "$env:APPDATA\uv"
Add-Content "$env:APPDATA\uv\uv.toml" 'python = "3.11"'
```

**Opción B — pyenv (Linux/macOS, si ya lo tenés instalado)**

Si usás pyenv, la versión global se configura con:

```bash
pyenv global 3.11.9
# Esto escribe en ~/.pyenv/version
cat ~/.pyenv/version  # verificar
```

uv respeta esta configuración automáticamente.

### Configurar el Python por proyecto

Dentro de la carpeta del proyecto, crear un archivo `.python-version`:

```bash
cd /ruta/al/proyecto
echo '3.11' > .python-version
```

Este archivo le indica a uv qué versión usar cuando opera dentro de esa carpeta. Es portable: si subís el proyecto a GitHub y otro desarrollador lo clona, obtendrá la misma versión.

**Verificar qué Python usaría uv en el directorio actual:**

```bash
uv python find
```

> **Jerarquía de resolución de Python en uv:**  
> `.python-version` del proyecto → `uv.toml` global → `pyenv global` → Python del sistema

<a id='5'></a>
## 🌿 5. Crear entornos virtuales con uv

### Crear un entorno básico

```bash
cd /ruta/al/proyecto
uv venv
```

Esto crea un directorio `.venv/` dentro del proyecto. Si hay un `.python-version` en la carpeta, uv lo respeta automáticamente. Si no, usa el Python global configurado.

### Crear un entorno con una versión específica (sin .python-version)

```bash
uv venv --python 3.11
```

### Activar el entorno

**Windows (PowerShell):**
```powershell
.venv\Scripts\activate
```

**Linux / macOS:**
```bash
source .venv/bin/activate
```

Una vez activado, el prompt del terminal muestra el nombre del entorno entre paréntesis: `(.venv) usuario@maquina:~$`

### Instalar paquetes

Con el entorno activado:

```bash
uv pip install numpy pandas matplotlib
```

O bien, sin necesidad de activar el entorno (uv lo detecta automáticamente si estás dentro de la carpeta del proyecto):

```bash
uv pip install numpy pandas matplotlib
```

### Verificar qué Python está usando el entorno

```bash
.venv/bin/python --version          # Linux / macOS
.venv\Scripts\python.exe --version  # Windows
```

### Guardar las dependencias del proyecto (reproducibilidad)

```bash
uv pip freeze > requirements.txt
```

Para recrear el entorno a partir de ese archivo en otra máquina:

```bash
uv venv
uv pip install -r requirements.txt
```

### ✎ Para pensar

- Creaste un entorno con `uv venv`. ¿Dónde queda guardado físicamente? ¿Qué hay adentro de `.venv/`?
- ¿Qué diferencia hace que el entorno esté activado o no cuando corrés `uv pip install`?
- Si borrás la carpeta `.venv/` y recreás el entorno con `uv pip install -r requirements.txt`, ¿obtenés exactamente el mismo entorno? ¿Qué garantiza eso?

In [ ]:
# ─── Espacio de práctica ──────────────────────────────────────────────────────
#
# Realizá el ciclo completo de un entorno nuevo desde cero:
#   1. Creá una carpeta llamada "practica-uv" en tu escritorio o donde prefieras
#   2. Creá un entorno virtual dentro de ella con uv venv
#   3. Activá el entorno
#   4. Instalá: jupyterlab requests
#   5. Exportá las dependencias con uv pip freeze > requirements.txt
#   6. Abrí el requirements.txt y verificá que ambas librerías están listadas
#
# Bonus: registrá el entorno como kernel de Jupyter con el nombre "practica-uv"
#        Verificá con `jupyter kernelspec list` que quedó registrado.
#        Cuando termines, dalo de baja y eliminá el .venv (ciclo completo).
#
# ─────────────────────────────────────────────────────────────────────────────

<a id='6'></a>
## 🔧 6. Registrar el entorno en VS Code

VS Code detecta entornos virtuales automáticamente si están en la carpeta del proyecto, pero conviene saber cómo forzar la selección.

### Método 1 — Detección automática

1. Abrir VS Code dentro de la carpeta del proyecto (`code .` desde la terminal, o `Archivo → Abrir carpeta`).
2. Abrir cualquier archivo `.py` o `.ipynb`.
3. En la barra inferior derecha, VS Code muestra el intérprete activo. Si detectó el `.venv`, aparecerá algo como `Python 3.11.x ('.venv': venv)`.

### Método 2 — Selección manual

1. Abrir la paleta de comandos: `Ctrl+Shift+P` (Windows/Linux) o `Cmd+Shift+P` (macOS).
2. Escribir `Python: Select Interpreter`.
3. Seleccionar la opción que muestre la ruta `.venv/bin/python` o `.venv\Scripts\python.exe`.

Si el entorno no aparece en la lista, elegir **"Enter interpreter path"** y escribir la ruta manualmente.

### Persistir la configuración en el proyecto

VS Code guarda la elección en `.vscode/settings.json`. Se puede establecer de forma explícita:

```json
{
  "python.defaultInterpreterPath": "${workspaceFolder}/.venv/bin/python"
}
```

En Windows:

```json
{
  "python.defaultInterpreterPath": "${workspaceFolder}\\.venv\\Scripts\\python.exe"
}
```

> **Buena práctica:** agregar `.vscode/settings.json` al repositorio Git del proyecto para que todos los colaboradores usen el mismo intérprete. El directorio `.venv/` en cambio siempre debe estar en `.gitignore`.

<a id='7'></a>
## 🧬 7. Registrar el entorno como kernel de Jupyter

### Analogía

Un kernel es como el motor de un auto. El notebook de Jupyter es el tablero de control — la interfaz desde donde escribís código, ves los resultados y navegás las celdas. Pero quien realmente ejecuta el código, procesa los datos y devuelve los resultados es el motor. Podés tener el mismo tablero conectado a motores distintos: uno con Python 3.10 y transformers, otro con Python 3.11 y fastapi. Cada motor es un entorno virtual diferente.

### Dónde vive esto en el mundo real

En un equipo de ciencia de datos, cada proyecto tiene su propio kernel registrado con su nombre. Así, al abrir un notebook, el selector de kernels muestra claramente a qué proyecto pertenece cada entorno — sin ambigüedad y sin mezclas. La gestión correcta del ciclo de vida de los kernels (registro al crear, baja al destruir el entorno) es lo que evita el problema del **kernel huérfano**: un kernel que aparece en la lista pero falla al ejecutar porque su entorno fue eliminado.

---

Para que un notebook de Jupyter pueda usar el entorno virtual que creamos, hay que registrar ese entorno como un kernel. Un kernel es el proceso Python que ejecuta el código de las celdas.

### Paso a paso

```bash
# 1. Ir a la carpeta del proyecto
cd /ruta/al/proyecto

# 2. Crear el entorno (si no existe)
uv venv

# 3. Activar el entorno
source .venv/bin/activate          # Linux / macOS
# .venv\Scripts\activate           # Windows PowerShell

# 4. Instalar ipykernel dentro del entorno
uv pip install ipykernel

# 5. Registrar el entorno como kernel de Jupyter
python -m ipykernel install --user \
  --name "nombre-del-proyecto" \
  --display-name "nombre-del-proyecto"
```

En Windows, el comando del paso 5 es en una sola línea:

```powershell
python -m ipykernel install --user --name "nombre-del-proyecto" --display-name "nombre-del-proyecto"
```

### Verificar que el kernel quedó registrado

```bash
jupyter kernelspec list
```

El nuevo kernel aparecerá en la lista. En VS Code, al abrir un `.ipynb` y hacer clic en el selector de kernel (esquina superior derecha), debe aparecer el kernel recién registrado.

### ¿Dónde se guardan los kernels?

```
macOS / Linux:  ~/Library/Jupyter/kernels/   o   ~/.local/share/jupyter/kernels/
Windows:        %APPDATA%\jupyter\kernels\
```

Cada kernel es una carpeta con un archivo `kernel.json` que contiene, entre otras cosas, la ruta al ejecutable Python del entorno.

### ✎ Para pensar

- ¿Qué pasa si registrás el kernel y después eliminás el `.venv/` sin darlo de baja primero? ¿Cómo se detecta ese problema?
- ¿Por qué el kernel se instala con `--user` y no en el entorno virtual mismo?

<a id='8'></a>
## 🧭 8. Workflow completo: crear, usar y limpiar sin dejar huérfanos

> **¿Qué es un kernel huérfano?**  
> Es un kernel registrado en Jupyter cuyo entorno virtual fue eliminado sin registrar la baja del kernel. El kernel aparece en la lista pero no puede ejecutar código porque el Python al que apunta ya no existe. Es el origen de errores confusos al abrir notebooks.

### Ciclo completo de un proyecto

**Fase 1 — Inicio del proyecto**

```bash
# Crear carpeta y moverse a ella
mkdir mi-proyecto && cd mi-proyecto

# Declarar la versión de Python del proyecto
echo '3.11' > .python-version

# Crear el entorno virtual
uv venv

# Activar
source .venv/bin/activate          # Linux / macOS
# .venv\Scripts\activate           # Windows

# Instalar dependencias
uv pip install ipykernel pandas matplotlib

# Registrar como kernel de Jupyter
python -m ipykernel install --user \
  --name "$(basename $PWD)" \
  --display-name "$(basename $PWD)"
```

**Fase 2 — Durante el trabajo**

```bash
# Activar el entorno cada vez que se retoma el trabajo
cd mi-proyecto
source .venv/bin/activate

# Agregar nuevas dependencias según se necesiten
uv pip install nueva-libreria

# Mantener el requirements.txt actualizado
uv pip freeze > requirements.txt
```

**Fase 3 — Cierre del proyecto (limpieza correcta)**

```bash
# PRIMERO: eliminar el kernel de Jupyter
jupyter kernelspec remove mi-proyecto

# SEGUNDO: eliminar el entorno virtual
rm -rf .venv

# Verificar que no quedó el kernel
jupyter kernelspec list
```

En Windows:

```powershell
# Eliminar kernel
jupyter kernelspec remove mi-proyecto

# Eliminar entorno
Remove-Item -Recurse -Force .venv
```

> **Regla mnemotécnica:** *«El kernel nace después del entorno y muere antes que él.»*  
> Siempre eliminar primero el kernel, después el `.venv`.

<a id='9'></a>
## 🗂️ 9. Verificación final del sistema

Los siguientes comandos permiten auditar el estado del entorno de desarrollo en cualquier momento.

### Auditoría completa (Linux / macOS)

```bash
echo "=== versión de uv ==="
uv --version

echo "=== Python activo en el shell ==="
python3 --version && which python3

echo "=== pythons gestionados por uv (instalados) ==="
uv python list --only-installed

echo "=== kernels de Jupyter registrados ==="
ls ~/Library/Jupyter/kernels/ 2>/dev/null \
  || ls ~/.local/share/jupyter/kernels/ 2>/dev/null \
  || echo "ningún kernel registrado"

echo "=== entornos virtuales activos en proyectos ==="
find ~ -maxdepth 5 -type d -name ".venv" 2>/dev/null | sort
```

### Auditoría completa (Windows PowerShell)

```powershell
Write-Host "=== versión de uv ==="
uv --version

Write-Host "=== Python activo ==="
python --version

Write-Host "=== pythons gestionados por uv ==="
uv python list --only-installed

Write-Host "=== kernels de Jupyter ==="
Get-ChildItem "$env:APPDATA\jupyter\kernels" -ErrorAction SilentlyContinue

Write-Host "=== entornos virtuales ==="
Get-ChildItem -Path $HOME -Recurse -Depth 5 -Filter ".venv" -Directory -ErrorAction SilentlyContinue
```

### Detectar kernels huérfanos

Un kernel es huérfano si la ruta al Python que indica su `kernel.json` no existe en el sistema.

```bash
# Linux / macOS: mostrar a qué Python apunta cada kernel registrado
for k in ~/Library/Jupyter/kernels/*/kernel.json; do
  nombre=$(dirname $k | xargs basename)
  ruta=$(python3 -c "import json; d=json.load(open('$k')); print(d['argv'][0])" 2>/dev/null)
  if [ -f "$ruta" ]; then
    echo "OK    $nombre → $ruta"
  else
    echo "HUÉR  $nombre → $ruta (no existe)"
  fi
done
```

---

## ⛰️ Resumen del workflow profesional

| Principio | Práctica |
|---|---|
| Un entorno por proyecto | Nunca compartir `.venv/` entre proyectos |
| La versión de Python va en el repo | Archivo `.python-version` commiteado |
| El `.venv/` nunca va al repo | Siempre en `.gitignore` |
| El kernel muere antes que el entorno | `jupyter kernelspec remove` antes de `rm -rf .venv` |
| Auditar periódicamente | Correr los comandos de esta sección cuando algo falla |

In [ ]:
# ─── VERIFICACIÓN RÁPIDA DESDE NOTEBOOK ───────────────────────────────────────
# Corré esta celda para ver el estado del entorno desde el que ejecuta este notebook.

import sys
import subprocess

print(f"Python:     {sys.version}")
print(f"Ejecutable: {sys.executable}")

# Ver si estamos dentro de un entorno virtual
en_venv = hasattr(sys, 'real_prefix') or (
    hasattr(sys, 'base_prefix') and sys.base_prefix != sys.prefix
)
print(f"En venv:    {'sí ✓' if en_venv else 'no — se recomienda activar un entorno virtual'}")

In [ ]:
# ─── LISTAR KERNELS DESDE NOTEBOOK ────────────────────────────────────────────
# Equivalente a 'jupyter kernelspec list' pero desde dentro del notebook.

import json
from pathlib import Path

kernel_dirs = [
    Path.home() / "Library" / "Jupyter" / "kernels",           # macOS
    Path.home() / ".local" / "share" / "jupyter" / "kernels",  # Linux
]

for kdir in kernel_dirs:
    if kdir.exists():
        kernels = list(kdir.iterdir())
        if kernels:
            print(f"Kernels en {kdir}:")
            for k in kernels:
                kfile = k / "kernel.json"
                if kfile.exists():
                    data = json.loads(kfile.read_text())
                    python_path = data.get("argv", [""])[0]
                    existe = "✓" if Path(python_path).exists() else "✗ HUÉRFANO"
                    print(f"  {existe}  {k.name:30s} → {python_path}")

---

*Guía de entornos Python con uv — material agnóstico de institución*  
Matías Barreto · [matiasbarreto.com](https://matiasbarreto.com)